In [ ]:
from datasets import load_dataset
from collections import defaultdict, Counter
import pickle
import math

In [ ]:
import ir_datasets


In [ ]:
print("Building doc_id to repo mapping for Python functions...")
doc_to_repo = {}
python_doc_ids = set()

for doc in dataset.docs_iter():
    if doc.language == 'python':
        doc_to_repo[doc.doc_id] = doc.repo
        python_doc_ids.add(doc.doc_id)

print(f"Found {len(python_doc_ids)} Python functions")

# Group qrels by repository for Python functions only
print("Grouping qrels by repository...")
qrels_by_repo = defaultdict(list)

for qrel in dataset.qrels_iter():
    # Only include qrels for Python functions
    if qrel.doc_id in python_doc_ids:
        repo = doc_to_repo[qrel.doc_id]
        qrels_by_repo[repo].append(qrel)

In [ ]:
sorted_repos = sorted(qrels_by_repo.items(), key=lambda x: len(x[1]), reverse=True)


In [ ]:
query_lookup = {}
doc_lookup = {}

for query in dataset.queries_iter():
    query_lookup[query.query_id] = query

# Build doc lookup (Python only)
for doc in dataset.docs_iter():
    if doc.language == 'python':
        doc_lookup[doc.doc_id] = doc

In [ ]:
entire_corpus=ir_datasets.load("codesearchnet")

print("Building doc_id to repo mapping for Python functions...")
entire_corpus_doc_to_repo = {}
entire_corpus_python_doc_ids = set()

for doc in entire_corpus.docs_iter():
    if doc.language == 'python':
        entire_corpus_doc_to_repo[doc.doc_id] = doc.repo
        entire_corpus_python_doc_ids.add(doc.doc_id)

print(f"Found {len(entire_corpus_python_doc_ids)} Python functions")

# Group documents by repositories
print("Grouping documents by repository...")
docs_by_repo = defaultdict(list)

for doc in entire_corpus.docs_iter():
    if doc.language == 'python':
        docs_by_repo[doc.repo].append(doc)

print(f"Found {len(docs_by_repo)} repositories")

# Sort repositories by number of documents (descending)
sorted_repos_by_doc_count = sorted(docs_by_repo.items(), key=lambda x: len(x[1]), reverse=True)

# Display top 10 repositories by document count
print("\nTop 10 repositories by document count:")
for i, (repo, docs) in enumerate(sorted_repos_by_doc_count[:10]):
    print(f"{i+1}. {repo}: {len(docs)} documents")


In [ ]:
# Convert sorted_repos to a dictionary for easy lookup
sorted_repos_dict = {}
for repo_name, qrels_list in sorted_repos:
    sorted_repos_dict[repo_name] = qrels_list

# Create the mapping - only for repos that have both docs and qrels
repo_mapping = {}

for repo_name, docs_list in sorted_repos_by_doc_count:
    # Only add if this repo also has qrels
    if repo_name in sorted_repos_dict and len(sorted_repos_dict[repo_name]) > 0:
        repo_mapping[repo_name] = {
            'documents': docs_list,
            'qrels': sorted_repos_dict[repo_name],
            'doc_count': len(docs_list),
            'qrels_count': len(sorted_repos_dict[repo_name])
        }

print(f"Found {len(repo_mapping)} repositories with both documents and qrels")


In [ ]:
len(sorted_repos_dict['apache/airflow'])

In [ ]:
i=0
for x in sorted_repos_dict:
    i+=1
    if len(sorted_repos_dict[x])==4:
        print(x)
        print(i)
        break

In [ ]:
sorted_repos_by_doc_count[2][0]


In [ ]:
for x in sorted_repos_dict.keys():
    if 'azure' in x.lower():
        print(x) 

In [ ]:
repo_mapping_sorted = dict(sorted(repo_mapping.items(), key=lambda item: item[1]['qrels_count'], reverse=True))

In [ ]:
# import pickle

# # Save all three data structures
# data_to_save = {
#     'query_lookup': query_lookup,
#     'doc_lookup': doc_lookup,
#     'repo_mapping_sorted': repo_mapping_sorted
# }

# with open('codesearchnet_data.pkl', 'wb') as f:
#     pickle.dump(data_to_save, f)

# print("Data saved to codesearchnet_data.pkl")

In [ ]:
with open('codesearchnet_data.pkl', 'rb') as f:
        loaded_data = pickle.load(f)
query_lookup = loaded_data['query_lookup']
doc_lookup = loaded_data['doc_lookup']
repo_mapping_sorted = loaded_data['repo_mapping_sorted']

In [ ]:

def deduplicate_qrels(qrels_list):
    """
    Process qrels list to get unique query_id, doc_id pairs with mode relevance scores.
    
    Args:
        qrels_list: List of qrel entries (can be accessed by index)
    
    Returns:
        List of deduplicated entries with mode relevance scores
    """
    # Group by (query_id, doc_id) pairs and collect relevance scores
    pairs_to_relevances = defaultdict(list)
    
    for i in range(len(qrels_list)):
        query_id = qrels_list[i][0]  # Access by index as mentioned
        doc_id = qrels_list[i][1]
        relevance = qrels_list[i][2]
        note = qrels_list[i][3] if len(qrels_list[i]) > 3 else ''
        
        pairs_to_relevances[(query_id, doc_id)].append({
            'relevance': relevance,
            'note': note,
            'original_entry': qrels_list[i]
        })
    
    # Create deduplicated results with mode relevance scores
    deduplicated_qrels = []
    
    for (query_id, doc_id), entries in pairs_to_relevances.items():
        # Get all relevance scores for this pair
        relevance_scores = [entry['relevance'] for entry in entries]
        
        # Find the mode (most common relevance score)
        relevance_counter = Counter(relevance_scores)
        mode_relevance = relevance_counter.most_common(1)[0][0]
        
        # Get the first entry that has the mode relevance (for note/other data)
        
        # Create the deduplicated entry
        deduplicated_qrels.append({
            'query_id': query_id,
            'doc_id': doc_id,
            'relevance': mode_relevance,
            'count': len(entries)  # How many duplicates were merged
        })

    query_to_docs = defaultdict(list)
    
    for entry in deduplicated_qrels:
        query_to_docs[entry['query_id']].append({
            'doc_id': entry['doc_id'],
            'relevance': entry['relevance'],
            'count': entry['count']
        })
    
    # Sort each query's docs by relevance (highest first)
    for query_id in query_to_docs:
        query_to_docs[query_id].sort(key=lambda x: x['relevance'], reverse=True)
    
    return dict(query_to_docs)
    


In [ ]:
repo_mapping_sorted['aloetesting/aloe_webdriver']['qrels']

In [ ]:
query_to_docs=deduplicate_qrels(repo_mapping_sorted['aloetesting/aloe_webdriver']['qrels'])

In [ ]:
query_to_docs

In [ ]:
doc_lookup[query_to_docs['44'][0]['doc_id']][3]

In [ ]:
doc_lookup[query_to_docs['44'][0]['doc_id']][2]

In [ ]:
i=0
for x in repo_mapping_sorted:
    i+=1
    print(repo_mapping_sorted[x]['qrels_count'])
    if repo_mapping_sorted[x]['qrels_count']<=4:
        break

In [ ]:
i

In [ ]:
repo_mapping_sorted.keys()

In [ ]:
REPO='ChrisCummins/labm8'

In [ ]:
repo_mapping_sorted[REPO]

In [ ]:
repo_mapping_sorted[REPO]['qrels']

In [ ]:
query_id=repo_mapping_sorted[REPO]['qrels'][3][0]
query=query_lookup[query_id][1]
print(query)

In [ ]:
for q in repo_mapping_sorted[REPO]['qrels']:
    if q[0]==query_id:
        print(q)
        print(doc_lookup[q[1]])
        

In [ ]:
doc_id=repo_mapping_sorted[REPO]['qrels'][2][1]
doc=doc_lookup[doc_id]
print(doc)



In [ ]:
query_to_docs=deduplicate_qrels(repo_mapping_sorted[REPO]['qrels'])

In [ ]:
from MCTS_cross_encoder_batch import RLEnhancedGraphRAG, main
from langchain_neo4j import Neo4jGraph
from sentence_transformers import SentenceTransformer


In [ ]:
graph_config = {
    "url": "bolt://localhost:7687",
    "username": "neo4j",
    "password": "nutanix_neo4j",
}

graph = Neo4jGraph(**graph_config)
embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")

rl_graphrag = RLEnhancedGraphRAG(
        graph=graph,
        embedding_model=embedding_model,
        max_iterations=100,
        reward_threshold=5.0,
        alpha=0.5,
        cross_encoder_model="BAAI/bge-reranker-v2-m3",
        top_k_children=15,
        top_k_references=10,
        reduce_top_k_flag=True,
        min_top_k_children=20,
        exploration_param=0.125
    )



In [ ]:
cypher_query = f"""
        MATCH (r:Repo {{name:'{'/home/ec2-user/CodeSearchNet/KelSolaar_Umbra'}'}})-[:CONTAINS]-(m:Module)
        RETURN count(m) as module_count
        """
module_result = graph.query(cypher_query)

In [ ]:
module_count = module_result[0]['module_count']
print(module_count)

In [ ]:
rl_graphrag.original_top_k_children=min(round(module_count/10)*5,200)+10
rl_graphrag.exploration_param=1/(8*math.sqrt(math.log(4*rl_graphrag.original_top_k_children)))

In [ ]:
print(rl_graphrag.exploration_param)
print(rl_graphrag.original_top_k_children)

In [ ]:
res=rl_graphrag.search(query,repo_name='/home/ec2-user/CodeSearchNet/KelSolaar_Umbra')


In [ ]:
res[3].keys()

In [ ]:
res[3]['node_data'].keys()


In [ ]:
res[3]['node_data']['class']+'.'+res[3]['node_data']['name']

In [ ]:
print(f"\nFound {len(res)} high-reward nodes:")
for i, node_info in enumerate(res, 1):
    print(
        f"{i}. {node_info['node_data'].get('name', 'Unknown')} "
        f"(Reward: {node_info['avg_reward']:.2f}, "
        f"Visits: {node_info['visit_count']})"
    )

In [ ]:
ground_truth_docs = query_to_docs.get(query_id, [])

# Create a mapping from function/method names to relevance scores
name_to_relevance = {}

for doc_info in ground_truth_docs:
    doc_id = doc_info['doc_id']
    relevance = int(doc_info['relevance'])
    
    # Get the document and extract the function name
    doc = doc_lookup[doc_id]
    func_name = doc[3]  # Function name is at index 3
    
    name_to_relevance[func_name] = relevance

# Extract names from retrieved results and create relevance vector
retrieved_relevances = []
print(name_to_relevance)
for i in range(min(10, len(res))):  # Only consider top 10
    relevance=0
    node_data = res[i]['node_data']
    
    # Extract the name based on node type
    if 'class' in node_data and node_data['class']:
        # This is a method - combine class and method name
        retrieved_name = f"{node_data['class']}.{node_data['name']}"
    else:
        # This is a function
        retrieved_name = node_data['name']
    print(retrieved_name)
    # Get relevance score (0 if not found in ground truth)
    # Check for exact match first
    if retrieved_name in name_to_relevance:
        relevance = name_to_relevance[retrieved_name]
    else:
        # If no exact match and this is a class node, check if it contains any ground truth methods
        node_type = node_data.get('node_type', '')
        if node_type == 'Class':
            class_name = node_data['name']
            # Look for any ground truth methods that belong to this class
            for gt_name, gt_relevance in name_to_relevance.items():
                if '.' in gt_name and gt_name.startswith(f"{class_name}."):
                    # Found a method in this class, use its relevance score
                    relevance = max(relevance, gt_relevance)  # Use highest relevance if multiple methods
    
    retrieved_relevances.append(relevance)

# Pad with zeros if we have less than 10 results
while len(retrieved_relevances) < 10:
    retrieved_relevances.append(0)

# Calculate NDCG@10
def calculate_dcg(relevances):
    dcg = 0.0
    for i, rel in enumerate(relevances):
        if rel > 0:
            dcg += rel / math.log2(i + 2)  # i+2 because positions are 1-indexed
    return dcg

# DCG for retrieved results
dcg = calculate_dcg(retrieved_relevances)

# IDCG - ideal DCG (sort ground truth by relevance)
ideal_relevances = sorted([int(doc_info['relevance']) for doc_info in ground_truth_docs], reverse=True)
ideal_relevances = ideal_relevances[:10]  # Top 10
while len(ideal_relevances) < 10:
    ideal_relevances.append(0)

idcg = calculate_dcg(ideal_relevances)

# NDCG@10
ndcg_10 = dcg / idcg if idcg > 0 else 0.0

# Calculate Recall@10
relevant_retrieved = sum(1 for rel in retrieved_relevances if rel > 0)
total_relevant = len(ground_truth_docs)
recall_10 = relevant_retrieved / total_relevant if total_relevant > 0 else 0.0

In [ ]:
recall_10

In [ ]:
graph.close()

In [ ]:
queue = [rl_graphrag.root_tree_node]

# Print header
print(f"{'Node ID':<120} {'Total_Reward':<12} {'Visits':<8} {'Sim_Reward':<12} {'Sim_Visits':<10} {'Child_Reward':<12} {'Child_Visits':<10} {'Retrieval_Score':<12} {'Cosine_Similarity':<12}")
print("-" * 100)

while queue:
    node = queue.pop(0)
    print(f"{node.graph_node_id:<120} {node.total_reward:<12.2f} {node.visit_count:<8} {node.simulation_reward:<12.2f} {node.simulation_visits:<10} {node.children_reward:<12.2f} {node.children_visits:<10} {node.retrieval_score:<12.2f} {node.cosine_similarity}")
    queue.extend(node.children)

In [ ]:
from add_embeddings_final import Code2Json
from code_2_text_prompts import summarisation_prompt, members_prompt, file_prompt


In [ ]:
URL = "local"
MODEL = "deepseek-ai/deepseek-coder-1.3b-instruct"

PROMPTS = {
    "base_prompt": summarisation_prompt,
    "file_prompt": file_prompt,
    "object_prompt": members_prompt
}


code2json = Code2Json(url=URL, model=MODEL, prompts=PROMPTS)


In [ ]:
user="""
**1. The provided Python code is a function named `make_report` that generates an HTML report based on a given date. The function takes an optional parameter `isodate` which defaults to 'today' if not provided. The func
tion fetches all jobs that have finished before midnight of the provided date, calculates their statistics, and generates a full report for each job. The report is then saved as an HTML file in the current director
y.

Here's a summary of the code:

**PURPOSE**
The purpose of this function is to generate an HTML report based on a given date.

**IMPLEMENTATION**
The function uses the `dbcmd`, `read`, `view_fullreport`, `html_parts`, `make_tabs`, and `PAGE_TEMPLATE` functions from the database, file handling, and template modules respectively. It fetches all jobs that have 
finished before midnight of the provided date, calculates their statistics, and generates a full report for each job. The report is then saved as an HTML file in the current directory.

**KEY FEATURES**
The function exhibits the following features:

1. It fetches all jobs that have finished before midnight of the provided date.
2. It calculates the statistics for each job.
3. It generates a full report for each job.
4. It saves the report as an HTML file in the current directory.

The function also exhibits the following features:

1. It uses the `dbcmd` function to execute a database command.  
2. It uses the `read` function to read a file from a specified directory.
3. It uses the `view_fullreport` function to view a full report.
4. It uses the `html_parts` function to split a string into HTML parts.
5. It uses the `make_tabs` function to create tabs for the report.
6. It uses the `PAGE_TEMPLATE` variable to generate the HTML page template.

**2.The provided Python code is used to generate a HTML string that represents a set of tabs. The function `make_tabs` takes three arguments: `tag_ids`, `tag_status`, and `tag_contents`.

The function begins by defining a template string `templ` that includes a `<div>` element with an `id` attribute of `"tabs"` and a `<ul>` element within it. The `<ul>` element contains a list of `<li>` elements, ea
ch representing a tab.

The function then initializes two empty lists: `lis` and `contents`. The `lis` list will contain the HTML code for each tab, and the `contents` list will contain the content for each tab.

The function then iterates over the `zip` of `tag_ids`, `tag_status`, and `tag_contents`. For each iteration, it checks the `status` of the current tag. If the status is 'complete', it appends a dot to the `mark` variable, otherwise, it appends an exclamation mark. It then appends the `mark` and `tag_id` to the `lis` list, and the `tag_content` to the `contents` list.

Finally, the function returns a formatted string that includes the `lis` list and the `contents` list, separated by newline characters. This formatted string is then used to generate the final HTML string.

**3.The function is designed to generate a set of tabs with unique IDs and statuses, and their corresponding content. The tabs are displayed in a tabbed interface, with each tab represented by a `<div>` element with a unique ID.
The function `html` is designed to convert a list of tuples describing a table into a HTML string. Here's a summary of the code:

**PURPOSE**
The function takes a list of tuples as input, where each tuple represents a row in the table. It then generates a table using the provided tuples and returns the HTML representation of the table.

**IMPLEMENTATION**
The function uses a Python feature called list comprehension to generate a list of strings, where each string is a row of the table. It then uses the `map` function to convert each tuple to a string. The `next` function is used to get the next value from the `tablecounter` generator, which is used to generate unique names for each table. The `HtmlTable` class is used to generate the HTML representation of the table.

**KEY FEATURES**
The function exhibits the following key features:

1. It generates a unique name for each table.
2. It converts the input list of tuples into a HTML string.
3. It uses a generator to generate unique names for each table. 
4. It uses the `HtmlTable` class to generate the HTML representation of the table.
"""

In [ ]:
file_prompt_2 = """
### Task: Create a Summary Report
You are a technical documentation specialist. Read the following function descriptions and create a concise summary report.

**Important:** 
Go thorugh each description and then create a DETIALED CONCISE summary of each description and then concatenate them.
In your summaries INCLUDE ALL variable names, function names, class names, etc. from the descriptions.
DO NOT REPEAT LINES IN OUTPUT

### Function Descriptions:



"""

In [ ]:
query = {
        "system_prompt": file_prompt,
        "user_prompt": user,
        "file_name":  "unknown",
        "file_path":  "unknown",
        "code_preview": "File-level description (no code)",
    }
    


In [ ]:
res=code2json.get_llm_response(query)

In [ ]:
print(res)

In [ ]:
input=file_prompt_2+"\n"+user+"\n"+"### Response:"

In [ ]:
print(input)

In [ ]:
res=code2json.llm.generate([input], code2json.sampling_params)

In [ ]:
print(res[0].outputs[0].text)

In [ ]:
messages = [
                {"role": "system", "content": file_prompt},
                {"role": "user", "content": user}
            ]
            
formatted_prompt = code2json.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

In [ ]:
print(formatted_prompt)

In [ ]:
res=code2json.llm.generate([formatted_prompt], code2json.sampling_params)

In [ ]:
print(res[0].outputs[0].text.strip())

In [ ]:
code=''' 
            '''

In [ ]:
system_prompts = [
    PROMPTS["base_prompt"],
    PROMPTS["object_prompt"]
]

user_prompt = code 

# Create queries for batch processing
query={
        "system_prompt": system_prompts[1],
        "user_prompt": user_prompt,
        "file_name": "unknown",
        "file_path": "unkown",
        "code_preview": "\n".join(code.split("\n")[:5]),
    }


# Get batch responses
responses = code2json.get_llm_response(query)

In [ ]:
responses 

In [ ]:
from sentence_transformers import SentenceTransformer
from langchain_neo4j import Neo4jGraph



In [ ]:
embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")
embedding_model.to('cuda') 

In [ ]:
res=embedding_model.encode(""" **Summary Report**

The following classes and methods are used to parse TrendMicro log files:

- TrendMicro log parser class (inherits from `dsv_parser`): 
  - Attributes: MIN_COLUMNS (minimum number of columns required), COLUMNS (list of column names)
  - Methods: `_CreateDictReader` (iterates over log lines, splits into values, checks column count), `_ParseTimestamp` (parses timestamp or falls back to date and time), `_ConvertToTimestamp` (converts date and time strings to timestamp)

- `OfficeScanWebReputationParser` (inherits from `TrendMicroBaseParser`): 
  - Attributes: COLUMNS (list of column names), MIN_COLUMNS (minimum number of columns required)
  - Methods: `ParseRow` (parses a row, takes `ParserMediator` and row data, produces `DateTimeValuesEvent`), `VerifyRow` (verifies row format, checks column count and timestamp)

- `OfficeScanVirusDetectionParser` (inherits from `TrendMicroBaseParser`): 
  - Attributes: NAME (parser name), DESCRIPTION (parser description), COLUMNS (list of column names), MIN_COLUMNS (minimum number of columns required)
  - Methods: `ParseRow` (parses a line, takes `parser_mediator` and `row`, produces `TrendMicroAVEventData` and `DateTimeValuesEvent`), `VerifyRow` (verifies row format, checks column count and timestamp)

These classes and methods are designed to parse and verify TrendMicro Office Scan log files, specifically Web Reputation and Virus Detection logs. The `OfficeScanWebReputationParser` and `OfficeScanVirusDetectionParser` classes handle the parsing and verification of log files, while the TrendMicro log parser class provides a more general solution for parsing TrendMicro log files.
""")

In [ ]:
print(res)

In [ ]:
url = "bolt://localhost:7687"
username = "neo4j"
password = "nutanix_neo4j"
graph = Neo4jGraph(url=url, username=username, password=password)

In [ ]:
graph.query(
    """
    Match (c:Module {name:"plaso.parsers.trendmicroav"})
    SET c.embedding = $embedding
    RETURN c
    """,
    {
        "embedding": res
    }
)

In [ ]:
member_descriptions = """  
Important Code Members:
lon_lat - main function that parses coordinate string and returns longitude/latitude tuple
value - input parameter containing space-separated coordinate pair as string
longitude - function call that converts first coordinate value to longitude format
latitude - function call that converts second coordinate value to latitude format

  """

graph.query(
    """
    MATCH (f:Function {name: $name})
    SET f.member_descriptions = $member_descriptions
    RETURN f
    """,
    {"name": "lon_lat", "member_descriptions": member_descriptions}
)

In [ ]:
res=graph.query(
    """
    MATCH (f:Function {name: $name})
    RETURN f.description, f.member_descriptions
    """,
    {"name": "lon_lat"})

In [ ]:
res[0]['f.member_descriptions']

In [ ]:
emb=embedding_model.encode(res[0]['f.description'] +'\n'+ res[0]['f.member_descriptions'])

In [ ]:
graph.query(
    """
    Match (c:Function {name:"lon_lat"})
    SET c.embedding = $embedding
    RETURN c
    """,
    {
        "embedding": emb
    }
)

In [ ]:
from FlagEmbedding import FlagReranker


In [ ]:
res=graph.query(
    """
   MATCH (c:Class {name: "MsSqlHook"})-[:HAS_METHOD]-(m {name:"get_conn"})
RETURN m.description, m.member_descriptions
    """,
)

In [ ]:
res

In [ ]:

cross_encoder = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=False)


In [ ]:
query

In [ ]:
cross_encoder.compute_score((query,res[0]['m.description'] + res[0]['m.member_descriptions']), normalize=True)

In [ ]:
import json
with open('codesearchnet_evaluation_results4.json', 'r') as f:
    mcts_results = json.load(f)

with open('codesearchnet_evaluation_results_vector_qwen3_8b_2.json', 'r') as f:
    vector_results = json.load(f)

mcts_repos = set(mcts_results['repositories'].keys())
vector_repos = set(vector_results['repositories'].keys())

# Find the 2 extra repositories in vector results
extra_repos = vector_repos - mcts_repos
print("Extra repositories in Vector eval:")
for repo in extra_repos:
    repo_data = vector_results['repositories'][repo]
    print(f"  {repo}: {len(repo_data['queries'])} queries, {repo_data['repo_stats']['total_queries']} total")

# Also check the overall_stats discrepancy
print(f"\nMCTS overall_stats says {mcts_results['overall_stats']['total_repositories']} repos")
print(f"But only {len(mcts_repos)} repos in repositories object")

print(f"\nVector overall_stats says {vector_results['overall_stats']['total_repositories']} repos") 
print(f"And has {len(vector_repos)} repos in repositories object ✅")

In [ ]:
import json

# Load vector results
with open('codesearchnet_evaluation_results_vector_qwen3_8b_2.json', 'r') as f:
    data = json.load(f)

# Repositories to exclude
exclude_repos = {'ChrisCummins/labm8', 'ulf1/oxyba'}

# Collect all scores
all_ndcg = []
all_recall = []
total_queries = 0

for repo_name, repo_data in data['repositories'].items():
    if repo_name in exclude_repos:
        continue
    
    # Get scores from each query in this repo
    for query_id, query_data in repo_data['queries'].items():
        all_ndcg.append(query_data['ndcg_10'])
        all_recall.append(query_data['recall_10'])
        total_queries += 1

# Compute averages
avg_ndcg = sum(all_ndcg) / len(all_ndcg) if all_ndcg else 0.0
avg_recall = sum(all_recall) / len(all_recall) if all_recall else 0.0

print(f"Excluding {exclude_repos}:")
print(f"Total queries: {total_queries}")
print(f"Overall NDCG@10: {avg_ndcg:.4f}")
print(f"Overall Recall@10: {avg_recall:.4f}")

# For comparison, show original numbers
print(f"\nOriginal (all repos):")
print(f"Total queries: {data['overall_stats']['total_queries']}")
print(f"Overall NDCG@10: {data['overall_stats']['overall_avg_ndcg_10']:.4f}")
print(f"Overall Recall@10: {data['overall_stats']['overall_avg_recall_10']:.4f}")